In [3]:
import io
import sys
import pandas as pd
import numpy as np

from langchain_ollama import ChatOllama

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.syntax import Syntax
from rich import box


console = Console()


# ============================================================
# Load Dataset
# ============================================================

df = pd.read_csv("dataset.csv")

console.print(
    "\n[bold green]Dataset loaded successfully[/bold green]\n"
)

table = Table(
    title="Dataset Preview",
    box=box.DOUBLE_EDGE
)

for col in df.columns:
    table.add_column(str(col))

for _, row in df.iterrows():
    table.add_row(*[str(i) for i in row])

console.print(table)


# ============================================================
# Ollama LLM
# ============================================================

# Make sure Ollama is running and llama3.1 is installed:
# ollama pull llama3.1

llm = ChatOllama(
    model="llama3.1",
    temperature=0
)


# ============================================================
# Agent 1 - Guiding Agent
# ============================================================

tasks = [
    "Display the entire dataset",

    "Print number of rows, number of columns and column names",

    "Identify and print quantitative and qualitative columns",

    "Detect missing values, print number of missing values and handle them",

    "Detect outliers, print high and low outliers and handle them using IQR",

    "Find duplicates and print number of duplicate rows",

    "Check the data types of all columns and convert columns to appropriate data types where necessary",

    "Detect highly skewed numerical columns and apply an appropriate transformation such as log1p where appropriate",

    "Detect columns with constant or near-zero variance and identify them as candidates for removal",
]


# ============================================================
# Agent 2 - Coding Agent
# ============================================================

def coder_agent(task):

    prompt = f"""
You are an expert Python data preprocessing engineer.

You are part of a multi-agent data preprocessing system.

The dataframe is already loaded and is named df.

Rules:

1. DO NOT reload the dataset.

2. Use pandas and numpy.

3. Return ONLY executable Python code.

4. DO NOT use markdown.

5. DO NOT use ```python.

6. DO NOT explain the code.

7. Any result that should be displayed must use print().

8. Modify df directly when preprocessing is required.

9. Numeric columns must be identified using:
   df.select_dtypes(include=np.number)

10. Categorical columns must be identified using:
    df.select_dtypes(include="object")

11. NEVER calculate df.mean() on the entire dataframe.

12. For missing numeric values:
    - Handle each numeric column separately.
    - Use the column mean only for numeric columns.

13. For missing categorical/object values:
    - Handle each categorical column separately.
    - Use "missing_value" for missing categorical values.

14. Do NOT use:
    df.fillna(df.mean(), inplace=True)

15. Do NOT use a fake column name such as:
    df['categorical_column']

16. For IQR outlier handling:
    - Identify numeric columns.
    - Calculate Q1 for each numeric column.
    - Calculate Q3 for each numeric column.
    - Calculate IQR = Q3 - Q1.
    - Lower bound = Q1 - 1.5 * IQR.
    - Upper bound = Q3 + 1.5 * IQR.
    - Print the number of low and high outliers.
    - Handle outliers using clipping rather than deleting rows.
    - Use:
      df[col] = df[col].clip(
          lower=lower_bound,
          upper=upper_bound
      )

17. For duplicates:
    - Print the number of duplicate rows.
    - Remove duplicates using:
      df.drop_duplicates(inplace=True)

18. For data type checking:
    - Print the current data type of every column.
    - Detect date-like columns and convert them to datetime when appropriate.
    - Detect numeric-looking columns and convert them to numeric when appropriate.
    - Do NOT convert SKU, ID, product codes, or identifiers into numeric columns merely because they contain numbers.
    - Preserve categorical and text columns.
    - Print the final data types after conversion.

19. For skewness:
    - Only analyze numerical columns.
    - Calculate skewness using:
      df[col].skew()
    - A column is considered highly skewed when:
      abs(skewness) > 1
    - Print the skewness of numerical columns.
    - For highly positively skewed numerical columns containing only non-negative values, apply:
      np.log1p(df[col])
    - Do NOT apply log1p to columns containing negative values.
    - Do NOT transform identifier columns.
    - Print which columns were transformed.
    - Do not transform a column more than once.

20. For constant and near-zero variance:
    - Only analyze numerical columns.
    - Identify constant columns where:
      df[col].nunique() <= 1
    - Identify near-zero variance columns using an appropriate threshold.
    - Print the constant columns.
    - Print the near-zero variance columns.
    - Do NOT automatically remove these columns.
    - Only identify them as candidates for removal.

21. Be careful with:
    - dates
    - SKU
    - ID
    - product codes
    - categorical columns
    - text columns

22. Do not perform mathematical operations on text, date, SKU, ID,
    product code, or categorical columns.

23. Before applying a transformation, verify that the column is numeric.

24. Do not create fake column names.

25. Clearly print what preprocessing was performed.

Task:
{task}
"""

    response = llm.invoke(prompt)

    code = response.content.strip()

    # Remove markdown fences if Ollama returns them
    code = code.replace("```python", "")
    code = code.replace("```", "")
    code = code.strip()

    console.print(
        "\n[bold yellow]Generated Code[/bold yellow]\n"
    )

    syntax = Syntax(
        code,
        "python",
        theme="monokai",
        line_numbers=False
    )

    console.print(syntax)

    return code


# ============================================================
# Agent 3 - Executor Agent
# ============================================================

def executor_agent(code):

    global df

    old_stdout = sys.stdout
    buffer = io.StringIO()

    try:

        local_vars = {
            "df": df,
            "np": np,
            "pd": pd
        }

        # Capture print output
        sys.stdout = buffer

        exec(
            code,
            {
                "pd": pd,
                "np": np
            },
            local_vars
        )

        # Restore stdout
        sys.stdout = old_stdout

        # Retrieve modified dataframe
        df = local_vars.get("df", df)

        output = buffer.getvalue()

        console.print(
            Panel(
                output if output else "Execution Completed",
                title="Execution Output",
                style="bold cyan on black"
            )
        )

        return True

    except Exception as e:

        sys.stdout = old_stdout

        console.print(
            Panel(
                f"{type(e).__name__}: {str(e)}",
                title="Execution Error",
                style="bold red"
            )
        )

        return False


# ============================================================
# Agent 4 - Checker Agent
# ============================================================

def checker_agent(iteration):

    if iteration < len(tasks):

        console.print(
            "\n[bold cyan]"
            "Checker Agent --> Pending Preprocessing"
            "[/bold cyan]"
        )

    else:

        console.print(
            "\n[bold green]"
            "Checker Agent --> Preprocessed Completely"
            "[/bold green]"
        )


# ============================================================
# Multi-Agent Execution
# ============================================================

for i, task in enumerate(tasks, start=1):

    console.print(
        f"\n[bold magenta]"
        f"====================================\n"
        f"Iteration {i}\n"
        f"===================================="
        f"[/bold magenta]"
    )

    console.print(
        f"[bold blue]"
        f"Guiding Agent --> {task}"
        f"[/bold blue]"
    )

    # --------------------------------------------------------
    # Coding Agent
    # --------------------------------------------------------

    code = coder_agent(task)

    # --------------------------------------------------------
    # Executor Agent
    # --------------------------------------------------------

    success = executor_agent(code)

    # --------------------------------------------------------
    # Checker Agent
    # --------------------------------------------------------

    checker_agent(i)

    if not success:

        console.print(
            "\n[bold red]"
            f"Iteration {i} failed. "
            "Continuing with the next iteration."
            "[/bold red]"
        )


# ============================================================
# Final Dataset Information
# ============================================================

console.print(
    "\n[bold green]Final Cleaned Dataset[/bold green]\n"
)

console.print(
    f"Rows: {len(df)}"
)

console.print(
    f"Columns: {len(df.columns)}"
)

console.print(
    f"Column names: {list(df.columns)}"
)


# ============================================================
# Final Cleaned Dataset Table
# ============================================================

final_table = Table(
    title="Processed Dataset",
    box=box.DOUBLE_EDGE
)

for col in df.columns:
    final_table.add_column(str(col))

for _, row in df.iterrows():
    final_table.add_row(
        *[str(i) for i in row]
    )

console.print(final_table)


# ============================================================
# Final Missing Value Check
# ============================================================

console.print(
    "\n[bold yellow]Final Missing Value Check[/bold yellow]"
)

missing_values = df.isnull().sum()

console.print(
    missing_values.to_string()
)


# ============================================================
# Final Duplicate Check
# ============================================================

console.print(
    "\n[bold yellow]Final Duplicate Check[/bold yellow]"
)

duplicate_count = df.duplicated().sum()

console.print(
    f"Duplicate rows remaining: {duplicate_count}"
)


# ============================================================
# Save Dataset
# ============================================================

output_file = "Final_Dataset.csv"

df.to_csv(
    output_file,
    index=False
)

console.print(
    f"\n[bold green]"
    f"Processed dataset saved to {output_file}"
    f"[/bold green]"
)

Dataset loaded successfully

                        Dataset Preview                        
╔════════════╤═════════╤══════════╤═══════════════╤═══════════╗
║ date       │ sku     │ location │ quantity_sold │ inventory ║
╟────────────┼─────────┼──────────┼───────────────┼───────────╢
║ 2024-01-01 │ SKU_101 │ Chennai  │ 52.0          │ 120       ║
║ 2024-01-02 │ SKU_101 │ Chennai  │ 49.0          │ 115       ║
║ 2024-01-03 │ SKU_101 │ Chennai  │ nan           │ 110       ║
║ 2024-01-04 │ SKU_101 │ Chennai  │ 55.0          │ 105       ║
║ 2024-01-05 │ SKU_101 │ Chennai  │ 60.0          │ 100       ║
║ 2024-01-06 │ SKU_101 │ Chennai  │ -5.0          │ 95        ║
║ 2024-01-07 │ SKU_101 │ Chennai  │ 58.0          │ 90        ║
║ 2024-01-08 │ SKU_101 │ Chennai  │ 62.0          │ 85        ║
║ 2024-01-09 │ SKU_101 │ Chennai  │ 59.0          │ 80        ║
║ 2024-01-10 │ SKU_101 │ Chennai  │ 57.0          │ 75        ║
║ 2024-01-11 │ SKU_101 │ Chennai  │ 300.0         │ 70        ║
║ 2024-01-12 │ SKU_101 │ Chennai  │ 56.0          │ 65        ║
║ 2024-01-13 │ SKU_101 │ Chennai  │ 54.0          │ 60        ║
║ 2024-01-14 │ SKU_101 │ Chennai  │ 2.0           │ 55        ║
║ 2024-01-15 │ SKU_101 │ Chennai  │ 53.0          │ 50        ║
║ 2024-01-16 │ SKU_101 │ Chennai  │ 61.0          │ 45        ║
║ 2024-01-17 │ SKU_101 │ Chennai  │ 63.0          │ 40        ║
║ 2024-01-18 │ SKU_101 │ Chennai  │ 65.0          │ 35        ║
║ 2024-01-19 │ SKU_101 │ Chennai  │ nan           │ 30        ║
║ 2024-01-20 │ SKU_101 │ Chennai  │ 64.0          │ 25        ║
║ 2024-01-21 │ SKU_101 │ Chennai  │ 70.0          │ 20        ║
║ 2024-01-22 │ SKU_101 │ Chennai  │ 72.0          │ 15        ║
║ 2024-01-23 │ SKU_101 │ Chennai  │ 68.0          │ 10        ║
║ 2024-01-24 │ SKU_101 │ Chennai  │ 500.0         │ 5         ║
║ 2024-01-25 │ SKU_101 │ Chennai  │ 66.0          │ 0         ║
║ 2024-01-26 │ SKU_101 │ Chennai  │ 67.0          │ 0         ║
║ 2024-01-27 │ SKU_101 │ Chennai  │ 69.0          │ 0         ║
║ 2024-01-28 │ SKU_101 │ Chennai  │ 71.0          │ 10        ║
║ 2024-01-29 │ SKU_101 │ Chennai  │ 73.0          │ 15        ║
║ 2024-01-30 │ SKU_101 │ Chennai  │ 74.0          │ 20        ║
╚════════════╧═════════╧══════════╧═══════════════╧═══════════╝

====================================
Iteration 1
====================================

Guiding Agent --> Display the entire dataset

Generated Code

...

Final Missing Value Check

date             0
sku              0
location         0
quantity_sold    0
inventory        0

Final Duplicate Check

Duplicate rows remaining: 0

Processed dataset saved to Final_Dataset.csv

In [1]:

import io
import sys
import pandas as pd
import numpy as np

from langchain_ollama import ChatOllama

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.syntax import Syntax
from rich import box


# ============================================================
# CONFIGURATION
# ============================================================

DATASET_FILE = "dataset.csv"
OUTPUT_FILE = "Final_Dataset.csv"

OLLAMA_MODEL = "llama3.1"


# ============================================================
# RICH CONSOLE
# ============================================================

console = Console()


# ============================================================
# LOAD DATASET
# ============================================================

try:
    df = pd.read_csv(DATASET_FILE)

    console.print(
        "\n[bold green]Dataset loaded successfully[/bold green]\n"
    )

except Exception as e:
    console.print(
        Panel(
            str(e),
            title="Dataset Loading Error",
            style="bold red"
        )
    )
    raise SystemExit


# ============================================================
# DISPLAY DATAFRAME
# ============================================================

def display_dataframe(dataframe, title="Dataset Preview"):

    table = Table(
        title=title,
        box=box.DOUBLE_EDGE
    )

    for col in dataframe.columns:
        table.add_column(str(col))

    for _, row in dataframe.iterrows():

        table.add_row(
            *[
                str(value)
                for value in row
            ]
        )

    console.print(table)


# ============================================================
# DISPLAY ORIGINAL DATASET
# ============================================================

display_dataframe(
    df,
    "Original Dataset"
)


# ============================================================
# OLLAMA
# ============================================================

llm = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=0
)


# ============================================================
# AGENT 1 - GUIDING AGENT
# ============================================================

tasks = [

    "Display the entire dataset",

    "Print the number of rows, number of columns and all column names",

    "Identify and print quantitative/numerical columns and qualitative/categorical columns",

    "Detect missing values, print the number of missing values for every column, and handle missing values appropriately",

    "Detect outliers in numerical columns using the IQR method, print the low and high outliers, and handle the outliers using IQR",

    "Find duplicate rows, print the number of duplicate rows, and remove duplicate rows",

    "Check the data types of all columns and convert columns to appropriate data types where necessary",

    "Detect highly skewed numerical columns and apply an appropriate transformation such as log1p where appropriate",

    "Detect columns with constant or near-zero variance and identify them as candidates for removal"
]


def guiding_agent():

    console.print(
        "\n[bold green]Guiding Agent[/bold green]"
    )

    console.print(
        f"Total preprocessing tasks: {len(tasks)}"
    )

    return tasks


# ============================================================
# AGENT 2 - CODING AGENT
# ============================================================

def coder_agent(task):

    console.print(
        "\n[bold yellow]Coding Agent[/bold yellow]"
    )

    prompt = f"""
You are an expert Python data preprocessing engineer.

You are working inside a multi-agent preprocessing system.

The pandas DataFrame is already loaded.

DataFrame name:
df

IMPORTANT RULES:

1. Do NOT reload the dataset.
2. Do NOT create a new main dataframe.
3. Modify df directly when preprocessing is required.
4. Use pandas and numpy only.
5. Return ONLY executable Python code.
6. Do NOT return Markdown.
7. Do NOT use code fences.
8. Do NOT provide explanations outside the code.
9. Use print() for information that should be displayed.
10. Do not import unnecessary libraries.
11. Numerical columns must be identified using:

    df.select_dtypes(include=np.number)

12. Categorical columns must be identified using:

    df.select_dtypes(include="object")

13. For missing numerical values, use the median.
14. For missing categorical values, use the mode.
15. For outliers, use the IQR method.
16. Do not delete columns unless the task explicitly requires it.
17. Preserve all important information.
18. Make the generated code executable directly against df.
19. Do not use input().
20. Do not use exit().
21. Do not use sys.exit().
22. Do not reload the CSV.
23. Do not create plots.
24. Print the result of the operation.

TASK:

{task}

Return ONLY valid Python code.
"""

    try:

        response = llm.invoke(prompt)

        code = response.content.strip()

        # ----------------------------------------------------
        # Remove Markdown code fences
        # ----------------------------------------------------

        if code.startswith("```python"):
            code = code[len("```python"):]

        elif code.startswith("```"):
            code = code[len("```"):]

        if code.endswith("```"):
            code = code[:-3]

        code = code.strip()

        console.print(
            "\n[bold yellow]Generated Code[/bold yellow]\n"
        )

        syntax = Syntax(
            code,
            "python",
            theme="monokai",
            line_numbers=True
        )

        console.print(syntax)

        return code

    except Exception as e:

        console.print(
            Panel(
                str(e),
                title="Coding Agent Error",
                style="bold red"
            )
        )

        return None


# ============================================================
# AGENT 3 - EXECUTOR AGENT
# ============================================================

def executor_agent(code):

    global df

    console.print(
        "\n[bold cyan]Executor Agent[/bold cyan]"
    )

    if not code:

        console.print(
            Panel(
                "No code was generated.",
                title="Execution Error",
                style="bold red"
            )
        )

        return False

    old_stdout = sys.stdout
    buffer = io.StringIO()

    try:

        local_vars = {
            "df": df,
            "np": np,
            "pd": pd
        }

        sys.stdout = buffer

        exec(
            code,
            {
                "__builtins__": __builtins__,
                "pd": pd,
                "np": np
            },
            local_vars
        )

        sys.stdout = old_stdout

        if "df" in local_vars:
            df = local_vars["df"]

        output = buffer.getvalue()

        if output.strip():

            console.print(
                Panel(
                    output,
                    title="Execution Output",
                    style="bold cyan"
                )
            )

        else:

            console.print(
                Panel(
                    "Execution completed successfully.",
                    title="Execution Output",
                    style="bold cyan"
                )
            )

        return True

    except Exception as e:

        sys.stdout = old_stdout

        console.print(
            Panel(
                str(e),
                title="Execution Error",
                style="bold red"
            )
        )

        return False


# ============================================================
# AGENT 4 - CHECKER AGENT
# ============================================================

def checker_agent(task):

    console.print(
        "\n[bold magenta]Checker Agent[/bold magenta]"
    )

    try:

        # ----------------------------------------------------
        # Dataset size
        # ----------------------------------------------------

        console.print(
            f"Rows    : {df.shape[0]}"
        )

        console.print(
            f"Columns : {df.shape[1]}"
        )

        # ----------------------------------------------------
        # Missing values
        # ----------------------------------------------------

        total_missing = df.isnull().sum().sum()

        console.print(
            f"Missing values: {total_missing}"
        )

        if total_missing == 0:

            console.print(
                "[bold green]✓ No missing values[/bold green]"
            )

        else:

            console.print(
                "[bold yellow]⚠ Missing values remain[/bold yellow]"
            )

        # ----------------------------------------------------
        # Duplicates
        # ----------------------------------------------------

        duplicates = df.duplicated().sum()

        console.print(
            f"Duplicate rows: {duplicates}"
        )

        if duplicates == 0:

            console.print(
                "[bold green]✓ No duplicate rows[/bold green]"
            )

        else:

            console.print(
                "[bold yellow]⚠ Duplicate rows remain[/bold yellow]"
            )

        # ----------------------------------------------------
        # Data types
        # ----------------------------------------------------

        console.print(
            "\n[bold cyan]Data Types[/bold cyan]"
        )

        for column in df.columns:

            console.print(
                f"{column}: {df[column].dtype}"
            )

        # ----------------------------------------------------
        # Numerical columns
        # ----------------------------------------------------

        numeric_columns = df.select_dtypes(
            include=np.number
        ).columns.tolist()

        console.print(
            f"\nNumerical columns: {numeric_columns}"
        )

        # ----------------------------------------------------
        # Categorical columns
        # ----------------------------------------------------

        categorical_columns = df.select_dtypes(
            include="object"
        ).columns.tolist()

        console.print(
            f"Categorical columns: {categorical_columns}"
        )

        # ----------------------------------------------------
        # Infinite values
        # ----------------------------------------------------

        numeric_df = df.select_dtypes(
            include=np.number
        )

        if not numeric_df.empty:

            infinite_count = np.isinf(
                numeric_df.to_numpy()
            ).sum()

        else:

            infinite_count = 0

        console.print(
            f"Infinite values: {infinite_count}"
        )

        # ----------------------------------------------------
        # Constant columns
        # ----------------------------------------------------

        constant_columns = [
            column
            for column in df.columns
            if df[column].nunique(dropna=False) <= 1
        ]

        if constant_columns:

            console.print(
                f"\n[bold yellow]"
                f"Constant columns: {constant_columns}"
                f"[/bold yellow]"
            )

        else:

            console.print(
                "\n[bold green]"
                "No constant columns detected."
                "[/bold green]"
            )

        # ----------------------------------------------------
        # Task result
        # ----------------------------------------------------

        console.print(
            "\n[bold green]"
            "✓ Checker Agent validation completed"
            "[/bold green]"
        )

        return True

    except Exception as e:

        console.print(
            Panel(
                str(e),
                title="Checker Agent Error",
                style="bold red"
            )
        )

        return False


# ============================================================
# START PIPELINE
# ============================================================

console.print(
    "\n[bold magenta]"
    "====================================================\n"
    "       MULTI-AGENT DATA PREPROCESSING SYSTEM\n"
    "===================================================="
    "[/bold magenta]\n"
)


# ============================================================
# GUIDING AGENT
# ============================================================

preprocessing_tasks = guiding_agent()


# ============================================================
# PROCESS EACH TASK
# ============================================================

for iteration, task in enumerate(
    preprocessing_tasks,
    start=1
):

    console.print(
        "\n[bold magenta]"
        "====================================================\n"
        f"                 ITERATION {iteration}\n"
        "===================================================="
        "[/bold magenta]"
    )

    # --------------------------------------------------------
    # Agent 1
    # --------------------------------------------------------

    console.print(
        f"\n[bold blue]"
        f"Guiding Agent --> {task}"
        f"[/bold blue]"
    )

    # --------------------------------------------------------
    # Agent 2
    # --------------------------------------------------------

    code = coder_agent(task)

    # --------------------------------------------------------
    # Agent 3
    # --------------------------------------------------------

    success = executor_agent(code)

    # --------------------------------------------------------
    # Agent 4
    # --------------------------------------------------------

    if success:

        checker_agent(task)

    else:

        console.print(
            "[bold red]"
            "Checker Agent --> Execution failed"
            "[/bold red]"
        )


# ============================================================
# FINAL VALIDATION
# ============================================================

console.print(
    "\n[bold magenta]"
    "====================================================\n"
    "                  FINAL VALIDATION\n"
    "===================================================="
    "[/bold magenta]"
)

console.print(
    f"\nFinal rows              : {df.shape[0]}"
)

console.print(
    f"Final columns           : {df.shape[1]}"
)

console.print(
    f"Final missing values    : {df.isnull().sum().sum()}"
)

console.print(
    f"Final duplicate rows    : {df.duplicated().sum()}"
)


# ============================================================
# FINAL DATASET
# ============================================================

console.print(
    "\n[bold green]Final Cleaned Dataset[/bold green]\n"
)

display_dataframe(
    df,
    "Processed Dataset"
)


# ============================================================
# SAVE DATASET
# ============================================================

try:

    df.to_csv(
        OUTPUT_FILE,
        index=False
    )

    console.print(
        f"\n[bold green]"
        f"Final dataset saved successfully:"
        f" {OUTPUT_FILE}"
        f"[/bold green]"
    )

except Exception as e:

    console.print(
        Panel(
            str(e),
            title="File Saving Error",
            style="bold red"
        )
    )


# ============================================================
# COMPLETED
# ============================================================

console.print(
    "\n[bold green]"
    "====================================================\n"
    "       DATA PREPROCESSING COMPLETED SUCCESSFULLY\n"
    "===================================================="
    "[/bold green]"
)


Dataset loaded successfully

                       Original Dataset                        
╔════════════╤═════════╤══════════╤═══════════════╤═══════════╗
║ date       │ sku     │ location │ quantity_sold │ inventory ║
╟────────────┼─────────┼──────────┼───────────────┼───────────╢
║ 2024-01-01 │ SKU_101 │ Chennai  │ 52.0          │ 120       ║
║ 2024-01-02 │ SKU_101 │ Chennai  │ 49.0          │ 115       ║
║ 2024-01-03 │ SKU_101 │ Chennai  │ nan           │ 110       ║
║ 2024-01-04 │ SKU_101 │ Chennai  │ 55.0          │ 105       ║
║ 2024-01-05 │ SKU_101 │ Chennai  │ 60.0          │ 100       ║
║ 2024-01-06 │ SKU_101 │ Chennai  │ -5.0          │ 95        ║
║ 2024-01-07 │ SKU_101 │ Chennai  │ 58.0          │ 90        ║
║ 2024-01-08 │ SKU_101 │ Chennai  │ 62.0          │ 85        ║
║ 2024-01-09 │ SKU_101 │ Chennai  │ 59.0          │ 80        ║
║ 2024-01-10 │ SKU_101 │ Chennai  │ 57.0          │ 75        ║
║ 2024-01-11 │ SKU_101 │ Chennai  │ 300.0         │ 70        ║
║ 2024-01-12 │ SKU_101 │ Chennai  │ 56.0          │ 65        ║
║ 2024-01-13 │ SKU_101 │ Chennai  │ 54.0          │ 60        ║
║ 2024-01-14 │ SKU_101 │ Chennai  │ 2.0           │ 55        ║
║ 2024-01-15 │ SKU_101 │ Chennai  │ 53.0          │ 50        ║
║ 2024-01-16 │ SKU_101 │ Chennai  │ 61.0          │ 45        ║
║ 2024-01-17 │ SKU_101 │ Chennai  │ 63.0          │ 40        ║
║ 2024-01-18 │ SKU_101 │ Chennai  │ 65.0          │ 35        ║
║ 2024-01-19 │ SKU_101 │ Chennai  │ nan           │ 30        ║
║ 2024-01-20 │ SKU_101 │ Chennai  │ 64.0          │ 25        ║
║ 2024-01-21 │ SKU_101 │ Chennai  │ 70.0          │ 20        ║
║ 2024-01-22 │ SKU_101 │ Chennai  │ 72.0          │ 15        ║
║ 2024-01-23 │ SKU_101 │ Chennai  │ 68.0          │ 10        ║
║ 2024-01-24 │ SKU_101 │ Chennai  │ 500.0         │ 5         ║
║ 2024-01-25 │ SKU_101 │ Chennai  │ 66.0          │ 0         ║
║ 2024-01-26 │ SKU_101 │ Chennai  │ 67.0          │ 0         ║
║ 2024-01-27 │ SKU_101 │ Chennai  │ 69.0          │ 0         ║
║ 2024-01-28 │ SKU_101 │ Chennai  │ 71.0          │ 10        ║
║ 2024-01-29 │ SKU_101 │ Chennai  │ 73.0          │ 15        ║
║ 2024-01-30 │ SKU_101 │ Chennai  │ 74.0          │ 20        ║
╚════════════╧═════════╧══════════╧═══════════════╧═══════════╝

====================================================
       MULTI-AGENT DATA PREPROCESSING SYSTEM
====================================================

Guiding Agent

Total preprocessing tasks: 9

...

Final duplicate rows    : 0

Final Cleaned Dataset

                       Processed Dataset                       
╔════════════╤═════════╤══════════╤═══════════════╤═══════════╗
║ date       │ sku     │ location │ quantity_sold │ inventory ║
╟────────────┼─────────┼──────────┼───────────────┼───────────╢
║ 2024-01-01 │ SKU_101 │ Chennai  │ 52.0          │ 120.0     ║
║ 2024-01-02 │ SKU_101 │ Chennai  │ 49.0          │ 115.0     ║
║ 2024-01-03 │ SKU_101 │ Chennai  │ 62.5          │ 110.0     ║
║ 2024-01-04 │ SKU_101 │ Chennai  │ 55.0          │ 105.0     ║
║ 2024-01-05 │ SKU_101 │ Chennai  │ 60.0          │ 100.0     ║
║ 2024-01-06 │ SKU_101 │ Chennai  │ 37.5          │ 95.0      ║
║ 2024-01-07 │ SKU_101 │ Chennai  │ 58.0          │ 90.0      ║
║ 2024-01-08 │ SKU_101 │ Chennai  │ 62.0          │ 85.0      ║
║ 2024-01-09 │ SKU_101 │ Chennai  │ 59.0          │ 80.0      ║
║ 2024-01-10 │ SKU_101 │ Chennai  │ 57.0          │ 75.0      ║
║ 2024-01-11 │ SKU_101 │ Chennai  │ 87.5          │ 70.0      ║
║ 2024-01-12 │ SKU_101 │ Chennai  │ 56.0          │ 65.0      ║
║ 2024-01-13 │ SKU_101 │ Chennai  │ 54.0          │ 60.0      ║
║ 2024-01-14 │ SKU_101 │ Chennai  │ 37.5          │ 55.0      ║
║ 2024-01-15 │ SKU_101 │ Chennai  │ 53.0          │ 50.0      ║
║ 2024-01-16 │ SKU_101 │ Chennai  │ 61.0          │ 45.0      ║
║ 2024-01-17 │ SKU_101 │ Chennai  │ 63.0          │ 40.0      ║
║ 2024-01-18 │ SKU_101 │ Chennai  │ 65.0          │ 35.0      ║
║ 2024-01-19 │ SKU_101 │ Chennai  │ 62.5          │ 30.0      ║
║ 2024-01-20 │ SKU_101 │ Chennai  │ 64.0          │ 25.0      ║
║ 2024-01-21 │ SKU_101 │ Chennai  │ 70.0          │ 20.0      ║
║ 2024-01-22 │ SKU_101 │ Chennai  │ 72.0          │ 15.0      ║
║ 2024-01-23 │ SKU_101 │ Chennai  │ 68.0          │ 10.0      ║
║ 2024-01-24 │ SKU_101 │ Chennai  │ 87.5          │ 5.0       ║
║ 2024-01-25 │ SKU_101 │ Chennai  │ 66.0          │ 0.0       ║
║ 2024-01-26 │ SKU_101 │ Chennai  │ 67.0          │ 0.0       ║
║ 2024-01-27 │ SKU_101 │ Chennai  │ 69.0          │ 0.0       ║
║ 2024-01-28 │ SKU_101 │ Chennai  │ 71.0          │ 10.0      ║
║ 2024-01-29 │ SKU_101 │ Chennai  │ 73.0          │ 15.0      ║
║ 2024-01-30 │ SKU_101 │ Chennai  │ 74.0          │ 20.0      ║
╚════════════╧═════════╧══════════╧═══════════════╧═══════════╝

Final dataset saved successfully: Final_Dataset.csv

====================================================
       DATA PREPROCESSING COMPLETED SUCCESSFULLY
====================================================